Extracción de la información relevante del Dataset de AI-KG.  Volcado a 2 csv:
1. Ids ; abstracts
2. ID ; entidades ; relaciones ; hlight (solo las palabras marcadas)

# 0. INICIALIZACION

In [1]:
import bz2
import shutil
import os
import xml.etree.ElementTree as ET
import csv
import html

In [ ]:
# folder = "DATA"
folder = "../data/raw/"

files = [
    'res1',
    'res2',
    'res3',
    'res4',
    'res5-1',
    'res5-2',
    'res6-1',
    'res6-2',
    'res7-1',
    'res7-2',
    'res8-1',
    'res8-2',
    'res8-3',
    'res8-4'
    ]
files_comp = [
    'res1.xml.bz2',
    'res2.xml.bz2',
    'res3.xml.bz2',
    'res4.xml.bz2',
    'res5-1.xml.bz2',
    'res5-2.xml.bz2',
    'res6-1.xml.bz2',
    'res6-2.xml.bz2',
    'res7-1.xml.bz2',
    'res7-2.xml.bz2',
    'res8-1.xml.bz2',
    'res8-2.xml.bz2',
    'res8-3.xml.bz2',
    'res8-4.xml.bz2'
    ]

# 1. DESCOMPRESION

In [3]:
# for file in files_comp:
for file in files:
    print(f"-"*40)
    print(f"Comienza descompresion de archivo: {file}")
    
    file_comp = f"{file}.xml.bz2"
    file_in = os.path.join(folder, file_comp)

    file_desc = f"{file}.xml"
    file_out = os.path.join(folder, file_desc)
    
    
    with bz2.open(file_in, "rb") as f_in:
        with open(file_out, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
            
    print(f"Archivo {file} descomprimido")

----------------------------------------
Comienza descompresion de archivo: res1
Archivo res1 descomprimido
----------------------------------------
Comienza descompresion de archivo: res2
Archivo res2 descomprimido
----------------------------------------
Comienza descompresion de archivo: res3
Archivo res3 descomprimido
----------------------------------------
Comienza descompresion de archivo: res4
Archivo res4 descomprimido
----------------------------------------
Comienza descompresion de archivo: res5-1
Archivo res5-1 descomprimido
----------------------------------------
Comienza descompresion de archivo: res5-2
Archivo res5-2 descomprimido
----------------------------------------
Comienza descompresion de archivo: res6-1
Archivo res6-1 descomprimido
----------------------------------------
Comienza descompresion de archivo: res6-2
Archivo res6-2 descomprimido
----------------------------------------
Comienza descompresion de archivo: res7-1
Archivo res7-1 descomprimido
--------

# 2. LIMPIEZA XML

Dentro del apartado hlights no reconoce <b></b> como un nuevo nodo dentro de hlights.    
Estas son las entidades del abstract que son las que tengo que identificar.  

Al hacer print aparece:  
<hlight>has generated an &lt;b class="match term0"&gt;information&lt;/b&gt; &lt;b class="match term1"&gt;overload&lt;/b&gt;</hlight>

EN VEZ DE:  
<hlight>has generated an <b class="match term0">information</b> <b class="match term1">overload</b></hlight>

In [3]:
def limpiar_xml(input_xml, clean_xml):
    context = ET.iterparse(input_xml, events=("start", "end"))
    root = None

    for event, elem in context:
        if root is None:
            root = elem  # guardar raíz

        if event == "end" and elem.tag == "hlight":
            # desescapar solo el contenido de hlight
            texto = "".join(elem.itertext())
            texto_unescaped = html.unescape(texto)

            # eliminar contenido previo
            elem.clear()

            # volver a insertar como XML (desescapado)
            try:
                wrapper = ET.fromstring(f"<root>{texto_unescaped}</root>")
                for child in wrapper:
                    elem.append(child)
                if wrapper.text:
                    elem.text = wrapper.text
            except ET.ParseError:
                elem.text = texto_unescaped  # fallback

    tree = ET.ElementTree(root)
    tree.write(clean_xml, encoding="utf-8", xml_declaration=True)

In [4]:
for file in files:
    print(f"-"*40)
    print(f"Comienza limpieza de archivo: {file}")
    
    file_xml = f"{file}.xml"
    file_in = os.path.join(folder, file_xml)

    file_clean = f"{file}_clean.xml"
    file_out = os.path.join(folder, file_clean)
    
    
    limpiar_xml(file_in, file_out)
            
    print(f"Archivo {file} limpio")

----------------------------------------
Comienza limpieza de archivo: res1
Archivo res1 limpio
----------------------------------------
Comienza limpieza de archivo: res2
Archivo res2 limpio
----------------------------------------
Comienza limpieza de archivo: res3
Archivo res3 limpio
----------------------------------------
Comienza limpieza de archivo: res4
Archivo res4 limpio
----------------------------------------
Comienza limpieza de archivo: res5-1
Archivo res5-1 limpio
----------------------------------------
Comienza limpieza de archivo: res5-2
Archivo res5-2 limpio
----------------------------------------
Comienza limpieza de archivo: res6-1
Archivo res6-1 limpio
----------------------------------------
Comienza limpieza de archivo: res6-2
Archivo res6-2 limpio
----------------------------------------
Comienza limpieza de archivo: res7-1
Archivo res7-1 limpio
----------------------------------------
Comienza limpieza de archivo: res7-2
Archivo res7-2 limpio
----------------

# 3. VOLCADO DE xml A csv

## 3.1 Extraer abstracts y ids y eliminar duplicados

In [2]:
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
def volcado_abs(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    data = []

    for text in root.findall(".//text"):
        text_id = text.get("id")
        abs_elem = text.find("abs")

        if text_id and abs_elem is not None:
            abs_text = abs_elem.text.strip() if abs_elem.text else ""
            data.append({
                "id": text_id,
                "abs": abs_text
            })

    df = pd.DataFrame(data)
    return df

In [29]:
file = "res1_clean"
folder = "../data/processed/xml/"


In [ ]:
df_abs = volcado_abs(f"{folder}{file}.xml")

In [27]:
df_abs

,id,abs
0,arXiv:1408.6930,The use of mobile devices in combination with ...
1,arXiv:2202.06248,Project ATHENA aims to develop an application ...
2,arXiv:2011.02260,With the explosive growth of online informatio...
3,arXiv:1703.09108,Only few digital libraries and reference manag...
4,arXiv:1303.6369,Information overload is a serious problem in m...
...,...,...
1217605,arXiv:1212.2467,In this paper we present a family of algorithm...
1217606,arXiv:1711.00789,Effective learning of asymmetric and local fea...
1217607,arXiv:2002.03355,Functional quantile regression (FQR) is a usef...
1217608,arXiv:1701.01634,Geophysical methods offer several key advantag...


In [ ]:
df_abs.shape

(1217610, 2)

In [ ]:
df_abs.drop_duplicates().shape

(537320, 2)

In [32]:
csv_folder = "../data/processed/csv/"
df_abs.drop_duplicates().to_csv(f"{csv_folder}{file}_abstracts.csv", index=False)

RESTO DE ARCHIVOS

In [3]:
files = [
    'res2',
    'res3',
    'res4',
    'res5-1',
    'res5-2',
    'res6-1',
    'res6-2',
    'res7-1',
    'res7-2',
    'res8-1',
    'res8-2',
    'res8-3',
    'res8-4'
    ]

In [4]:
folder = "../data/processed/xml/"
csv_folder = "../data/processed/csv/"

In [5]:
for file in files:
    print(f"Comienza volcado para archivo: {file}")
    df_abs = volcado_abs(f"{folder}{file}_clean.xml")
    df_abs.drop_duplicates().to_csv(f"{csv_folder}{file}_abstracts.csv", index=False)
    print(f"Finaliza volcado para archivo: {file}")

Comienza volcado para archivo: res2
Finaliza volcado para archivo: res2
Comienza volcado para archivo: res3
Finaliza volcado para archivo: res3
Comienza volcado para archivo: res4
Finaliza volcado para archivo: res4
Comienza volcado para archivo: res5-1
Finaliza volcado para archivo: res5-1
Comienza volcado para archivo: res5-2
Finaliza volcado para archivo: res5-2
Comienza volcado para archivo: res6-1
Finaliza volcado para archivo: res6-1
Comienza volcado para archivo: res6-2
Finaliza volcado para archivo: res6-2
Comienza volcado para archivo: res7-1
Finaliza volcado para archivo: res7-1
Comienza volcado para archivo: res7-2
Finaliza volcado para archivo: res7-2
Comienza volcado para archivo: res8-1
Finaliza volcado para archivo: res8-1
Comienza volcado para archivo: res8-2
Finaliza volcado para archivo: res8-2
Comienza volcado para archivo: res8-3
Finaliza volcado para archivo: res8-3
Comienza volcado para archivo: res8-4
Finaliza volcado para archivo: res8-4


## 3.1 Extraer relaciones, entidades, ids y hblight (elementos detectados en el abstract)

In [3]:
def extraer_b_tags(text_elem):
    hlight = text_elem.find("hlight")
    if hlight is None:
        return ""

    resultados = []

    # for b in hlight.findall(".//b"):
    for b in hlight.findall("b"):
        resultados.append(b.text.strip())

    return "|".join(resultados)


In [4]:
def extraer_hlight_completo(text_elem):
    hlight = text_elem.find("hlight")
    if hlight is None:
        return ""

    return "".join(hlight.itertext()).strip()


In [7]:
import xml.etree.ElementTree as ET
import pandas as pd

def xml_to_df_triples(xml_file):
    data = []

    context = ET.iterparse(xml_file, events=("end",))

    for event, elem in context:
        if elem.tag == "triple":

            e1 = elem.get("e1")
            e2 = elem.get("e2")

            rels = []
            for r in elem.findall("./rels/r"):
                rels.append(r.text)

            # recorrer todos los text dentro del triple
            for text_elem in elem.findall("text"):
                text_id = text_elem.get("id")

                hlight_b = extraer_b_tags(text_elem)
                # hlight_full = extraer_hlight_completo(elem)

                data.append({
                    "entidades": (e1, e2),
                    "rels": rels,
                    "id": text_id,
                    # "hlight_full": hlight_full,
                    "hlight_b": hlight_b,
                })

    df = pd.DataFrame(data)
    return df


In [5]:
file = "res1_clean"
folder = "../data/processed/xml/"


In [ ]:
df = xml_to_df_triples(f"{folder}{file}.xml")
print(df.head())

In [25]:
df.shape

(1217610, 4)

In [39]:
csv_folder = "../data/processed/csv/"
df.to_csv(f"{csv_folder}{file}_rels_entis.csv", index=False)

RESTO DE ARCHIVOS

In [9]:
files = [
    'res2',
    'res3',
    'res4',
    'res5-1',
    'res5-2',
    'res6-1',
    'res6-2',
    'res7-1',
    'res7-2',
    'res8-1',
    'res8-2',
    'res8-3',
    'res8-4'
    ]

In [10]:
folder = "../data/processed/xml/"
csv_folder = "../data/processed/csv/"

In [11]:
for file in files:
    print(f"Comienza volcado para archivo: {file}")
    df = xml_to_df_triples(f"{folder}{file}_clean.xml")
    df.to_csv(f"{csv_folder}{file}_rels_entis.csv", index=False)
    print(f"Finaliza volcado para archivo: {file}")

Comienza volcado para archivo: res2
Finaliza volcado para archivo: res2
Comienza volcado para archivo: res3
Finaliza volcado para archivo: res3
Comienza volcado para archivo: res4
Finaliza volcado para archivo: res4
Comienza volcado para archivo: res5-1
Finaliza volcado para archivo: res5-1
Comienza volcado para archivo: res5-2
Finaliza volcado para archivo: res5-2
Comienza volcado para archivo: res6-1
Finaliza volcado para archivo: res6-1
Comienza volcado para archivo: res6-2
Finaliza volcado para archivo: res6-2
Comienza volcado para archivo: res7-1
Finaliza volcado para archivo: res7-1
Comienza volcado para archivo: res7-2
Finaliza volcado para archivo: res7-2
Comienza volcado para archivo: res8-1
Finaliza volcado para archivo: res8-1
Comienza volcado para archivo: res8-2
Finaliza volcado para archivo: res8-2
Comienza volcado para archivo: res8-3
Finaliza volcado para archivo: res8-3
Comienza volcado para archivo: res8-4
Finaliza volcado para archivo: res8-4


# 4. Creacion subset 1000 abstracts

Para las pruebas voy a coger los 1000 primeros abstracts en 1 csv, y en otro voy a poner todas las entidades/relaciones de estos que aparecen en todo el dataset

In [20]:
files = [
    'res1',
    'res2',
    'res3',
    'res4',
    'res5-1',
    'res5-2',
    'res6-1',
    'res6-2',
    'res7-1',
    'res7-2',
    'res8-1',
    'res8-2',
    'res8-3',
    'res8-4'
    ]

In [16]:
csv_folder = "../data/processed/csv/"
csv_abs = "res1_abstracts.csv"

abstracts_subset = pd.read_csv(f"{csv_folder}{csv_abs}", header=0, nrows=1000)
ids_subset = abstracts_subset["id"]

In [21]:
df_rels_ent_sub = pd.DataFrame(columns=["entidades", "rels", "id", "hlight_b"])
for file in files:
    print(f"Comienza archivo: {file}")
    df = pd.read_csv(f"{csv_folder}{file}_rels_entis.csv")
    df_sub = df[df["id"].isin(ids_subset)]
    df_rels_ent_sub = pd.concat([df_rels_ent_sub, df_sub])
    print(f"Archivo: {file} CONCATENADO")
    print("-"*30)

Comienza archivo: res1
Archivo: res1 CONCATENADO
------------------------------
Comienza archivo: res2
Archivo: res2 CONCATENADO
------------------------------
Comienza archivo: res3
Archivo: res3 CONCATENADO
------------------------------
Comienza archivo: res4
Archivo: res4 CONCATENADO
------------------------------
Comienza archivo: res5-1
Archivo: res5-1 CONCATENADO
------------------------------
Comienza archivo: res5-2
Archivo: res5-2 CONCATENADO
------------------------------
Comienza archivo: res6-1
Archivo: res6-1 CONCATENADO
------------------------------
Comienza archivo: res6-2
Archivo: res6-2 CONCATENADO
------------------------------
Comienza archivo: res7-1
Archivo: res7-1 CONCATENADO
------------------------------
Comienza archivo: res7-2
Archivo: res7-2 CONCATENADO
------------------------------
Comienza archivo: res8-1
Archivo: res8-1 CONCATENADO
------------------------------
Comienza archivo: res8-2
Archivo: res8-2 CONCATENADO
------------------------------
Comienza

In [27]:
abstracts_subset.sort_values("id").to_csv(f"{csv_folder}1000_abs_subset.csv", index=False)
df_rels_ent_sub.sort_values("id").to_csv(f"{csv_folder}1000_rels_entis_subset.csv", index=False)

In [ ]:
df.to_csv(f"{csv_folder}{file}_rels_entis.csv", index=False)